# dcgan-wrapper-netG-netD — ex2: wrapper train()/eval() propagation and state_dict round trip

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dcgan-wrapper-netG-netD`. Running the final beacon cell reports progress against the `Generative: DCGAN netG+netD wrapper` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: DCGAN netG+netD wrapper` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dcgan-wrapper-netG-netD`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dcgan-wrapper-netG-netD"
DD_SUBTOPIC = "Generative: DCGAN netG+netD wrapper"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## DCGAN wrapper `train()` / `eval()` propagation — quick refresher

`nn.Module.train(mode=True)` recursively switches `self.training = mode` for every submodule. For DCGAN that means `wrapper.train()` flips `netG`, `netD`, AND every BatchNorm/Dropout buried inside them — all in one call.

**Why this matters.** `nn.BatchNorm2d` uses BATCH statistics in train mode and FROZEN running statistics in eval mode. If you only flip the wrapper to `eval()` and one of the BN layers stays in train mode, your D will see a different distribution on identical input — a silent correctness bug that's hard to debug.

**`state_dict()` round-trips both subnets.** `wrapper.state_dict()` returns a flat dict keyed by qualified name (`netG.0.weight`, `netD.2.bias`, etc.). `wrapper.load_state_dict(sd)` restores both subnets at once. Saving a checkpoint = one call, not two.

### Exercise 2 — wrapper train()/eval() propagation and state_dict round trip

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the wrapper's compound semantics by building it, toggling `train()`/`eval()` and verifying BOTH subnets (incl. BatchNorm submodules) follow, then save+load `state_dict()` and verify a numerical round trip on both subnets.
> Keywords: dcgan, wrapper, train, eval, state_dict, batchnorm
> ```

**KCs targeted:** `wrapper-mode-propagates-to-both-subnets`, `state-dict-round-trips-both-subnets`

Implement `ex2_wrapper_lifecycle(generator, discriminator)`. The drill exercises the OPERATIONAL semantics the wrapper module enables — beyond just holding attributes:

1. Build the wrapper class (same shape as ex1: `nn.Module` subclass with `netG` and `netD` as submodules, no `forward`).
2. Instantiate it as `wrapper = DCGAN(generator, discriminator)`.
3. **Toggle to eval mode** on the wrapper: `wrapper.eval()`.
4. Snapshot the state dict: `sd_before = wrapper.state_dict()`. (Don't clone keys; do clone values to insulate from later writes.) Implement as: `sd_before = {k: v.detach().clone() for k, v in wrapper.state_dict().items()}`.
5. **Mutate every parameter in-place** (to prove the round trip recovers): `for p in wrapper.parameters(): p.data.add_(1.0)`.
6. **Load the snapshot back:** `wrapper.load_state_dict(sd_before)`.
7. Return the tuple `(wrapper, sd_before)`.

The test checks four invariants on the returned wrapper:
(a) `wrapper.training is False` (eval propagated to wrapper).
(b) `wrapper.netG.training is False` AND `wrapper.netD.training is False` (eval propagated to BOTH subnets).
(c) Every BatchNorm submodule inside either subnet has `m.training is False`.
(d) After load_state_dict, every parameter is bit-identical to the snapshot.

In [ ]:
def ex2_wrapper_lifecycle(generator: nn.Module, discriminator: nn.Module):
    """Build wrapper, eval(), snapshot, mutate, restore. Return (wrapper, snapshot)."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn

    # Build a generator with a BatchNorm so we can probe propagation.
    gen = nn.Sequential(
        nn.Linear(8, 16),
        nn.BatchNorm1d(16),
        nn.ReLU(),
        nn.Linear(16, 4),
    )
    disc = nn.Sequential(
        nn.Linear(4, 16),
        nn.BatchNorm1d(16),
        nn.LeakyReLU(),
        nn.Linear(16, 1),
    )

    wrapper, sd_before = ex2_wrapper_lifecycle(gen, disc)
    assert isinstance(wrapper, nn.Module), f'expected nn.Module, got {type(wrapper)}'
    assert hasattr(wrapper, 'netG') and hasattr(wrapper, 'netD'), 'wrapper must have netG/netD attrs'
    assert wrapper.netG is gen and wrapper.netD is disc, 'subnets must be the same instances'

    # (a) wrapper in eval mode.
    assert wrapper.training is False, f'wrapper.training={wrapper.training}, expected False'
    # (b) Both subnets in eval mode.
    assert wrapper.netG.training is False, f'netG.training={wrapper.netG.training}, expected False'
    assert wrapper.netD.training is False, f'netD.training={wrapper.netD.training}, expected False'

    # (c) Every BatchNorm inside either subnet in eval mode.
    bn_count = 0
    for m in wrapper.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            bn_count += 1
            assert m.training is False, f'BN {m} still in train mode after wrapper.eval()'
    assert bn_count >= 2, f'expected at least 2 BN layers (one per subnet), got {bn_count}'

    # (d) Snapshot round-trip — every param bit-identical to sd_before.
    sd_after = wrapper.state_dict()
    assert set(sd_after.keys()) == set(sd_before.keys()), 'state_dict keys must match'
    for k in sd_before:
        assert t.equal(sd_after[k], sd_before[k]), (
            f'state_dict round-trip failed for {k!r} — '
            f'max diff {(sd_after[k] - sd_before[k]).abs().max().item():.6f}.  '
            f'Did you call load_state_dict at the end?'
        )

    # Snapshot keys must span BOTH subnets — confirms state_dict captured both.
    netG_keys = [k for k in sd_before if k.startswith('netG.')]
    netD_keys = [k for k in sd_before if k.startswith('netD.')]
    assert len(netG_keys) > 0, f'snapshot has no netG.* keys: {list(sd_before)}'
    assert len(netD_keys) > 0, f'snapshot has no netD.* keys: {list(sd_before)}'

    # Toggle back to train mode — both subnets must follow.
    wrapper.train()
    assert wrapper.training is True
    assert wrapper.netG.training is True and wrapper.netD.training is True
    for m in wrapper.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            assert m.training is True
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_wrapper_lifecycle(generator: nn.Module, discriminator: nn.Module):
    class DCGAN(nn.Module):
        def __init__(self, netG, netD):
            super().__init__()
            self.netG = netG
            self.netD = netD
    wrapper = DCGAN(generator, discriminator)
    wrapper.eval()
    sd_before = {k: v.detach().clone() for k, v in wrapper.state_dict().items()}
    for p in wrapper.parameters():
        p.data.add_(1.0)
    wrapper.load_state_dict(sd_before)
    return wrapper, sd_before
```

**`eval()` is recursive by design.** `nn.Module.eval()` calls `self.train(False)`, which sets `self.training = False` AND iterates over `self.children()` calling `.train(False)` on each — recursively. Same for `.train(True)`. The wrapper inherits this for free; you do nothing special.

**Clone values in the snapshot.** `wrapper.state_dict()` returns tensors that ALIAS the live parameters. If you mutate parameters AFTER taking the snapshot but BEFORE cloning, your snapshot mutates too — the round trip becomes a no-op. Cloning at snapshot time decouples them.

**`load_state_dict` is in-place.** It copies values into the existing parameter tensors — it does NOT rebind the parameters. Any external references to `wrapper.netG[0].weight` still point at the same tensor object, now holding the restored values. Critical for optimizer state preservation.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()